# QA Generation (eval set)

In [ ]:
# --- Google Colab (use when running in Colab) ---
# from google.colab import drive
# drive.mount("/content/drive")

# --- Local WSL2 (use when running locally) ---
# No drive mount needed; data is accessed via /mnt/c/...

In [ ]:
!pip -q install -U unsloth peft accelerate bitsandbytes sentence-transformers chromadb \
    evaluate sacrebleu rouge-score nltk bert_score groq

In [ ]:
import os
os.environ["UNSLOTH_COMPILE_DISABLE"] = "1"
os.environ["TORCHDYNAMO_DISABLE"] = "1"
os.environ["TORCH_COMPILE_DISABLE"] = "1"


In [ ]:
import json, re, time
import numpy as np
import pandas as pd
import torch

SEED = 42
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

SUBJECT = "fcs"   # "fsd" | "fcs" | "dma"

# --- Google Colab (use when running in Colab) ---
# BASE_DIR = "/content/drive/MyDrive/Colab Notebooks/agentic-ai-tutor"

# --- Local WSL2 (use when running locally) ---
BASE_DIR = "/mnt/c/Users/nastt/Documents/agentic-ai-tutor/backend/data"

BASE_MODEL_NAME = "microsoft/Phi-4-mini-instruct"
MAX_SEQ_LEN = 2048
LOAD_IN_4BIT = True

ADAPTER_DIR = f"{BASE_DIR}/adapters/lora_phi4mini_{SUBJECT.upper()}"
EVAL_JSONL_PATH = f"{BASE_DIR}/eval_qa/{SUBJECT}_eval_OK.jsonl"

# RAG (chunk index + chroma dir)
CHUNK_CSV_PATH = f"{BASE_DIR}/rag/{SUBJECT}/chunk_index.csv"
CHROMA_DIR = f"{BASE_DIR}/rag/{SUBJECT}/chroma_store"
EMBED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

TOP_K = 6
MAX_CONTEXT_TOKENS = 1400

SYSTEM = (
    "You are a helpful tutor. Answer clearly and step-by-step when appropriate. "
    "Output ONLY the answer to the question. "
    "Do NOT include repository names, metadata tags, or unrelated code. "
    "If code is required, output only minimal code relevant to the question."
)

In [ ]:
def load_eval_jsonl(path: str, limit: int | None = None, seed: int = 42) -> pd.DataFrame:
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            obj = json.loads(line)
            q = obj.get("question")
            a = obj.get("answer")  # reference answer
            if q and a:
                rows.append({"question": q, "reference": a})
    df = pd.DataFrame(rows)
    if df.empty:
        raise ValueError("Eval file is empty or missing {question, answer} fields.")
    if limit is not None:
        df = df.sample(n=min(limit, len(df)), random_state=seed).reset_index(drop=True)
    return df

assert os.path.exists(EVAL_JSONL_PATH), f"Missing eval set: {EVAL_JSONL_PATH}"
df_eval = load_eval_jsonl(EVAL_JSONL_PATH, limit=None)
print("Eval size:", len(df_eval))
df_eval.head(3)


# Load SLM + LoRA

In [ ]:
import unsloth
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template
from peft import PeftModel

# Base
base_model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=LOAD_IN_4BIT,
    attn_implementation="sdpa"
)
tokenizer = get_chat_template(tokenizer, chat_template="chatml")
FastLanguageModel.for_inference(base_model)

# LoRA model
lora_base, _ = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=LOAD_IN_4BIT,
    attn_implementation="sdpa"
)
lora_model = PeftModel.from_pretrained(lora_base, ADAPTER_DIR)
FastLanguageModel.for_inference(lora_model)

print("Loaded base + lora.")


# RAG retrieval (Chroma)

In [ ]:
import chromadb
from chromadb.config import Settings
from sentence_transformers import SentenceTransformer

def safe_str(x) -> str:
    if x is None:
        return ""
    return x if isinstance(x, str) else str(x)

def normalize_text(s: str) -> str:
    s = safe_str(s).strip()
    s = re.sub(r"\s+", " ", s)
    return s

def approximate_count_tokens(text: str) -> int:
    text = safe_str(text)
    return 0 if not text else max(1, int(len(text) / 4))

def truncate_to_token_budget(hits, max_tokens: int):
    used = []
    parts = []
    total = 0
    for h in hits:
        t = int(h.get("tokens", approximate_count_tokens(h.get("text", ""))))
        if used and total + t > max_tokens:
            break
        parts.append(h["text"].strip())
        used.append(h)
        total += t
        if total >= max_tokens:
            break
    return "\n\n---\n\n".join([p for p in parts if p]), used

# Initialize ChromaDB client and collection once
os.makedirs(CHROMA_DIR, exist_ok=True)
_chroma_client = chromadb.PersistentClient(
    path=CHROMA_DIR,
    settings=Settings(anonymized_telemetry=False),
)
_chroma_collection = _chroma_client.get_or_create_collection(
    name="rag",
    metadata={"hnsw:space": "cosine"},
)

embedder = SentenceTransformer(EMBED_MODEL_NAME)

def retrieve_chunks(query: str, top_k: int = TOP_K):
    q = normalize_text(query)
    q_emb = embedder.encode([q], normalize_embeddings=True).astype(np.float32).tolist()[0]
    res = _chroma_collection.query(
        query_embeddings=[q_emb],
        n_results=top_k,
        include=["documents", "metadatas", "distances"],
    )
    docs = res.get("documents", [[]])[0]
    metas = res.get("metadatas", [[]])[0]
    dists = res.get("distances", [[]])[0]

    hits = []
    for doc, meta, dist in zip(docs, metas, dists):
        meta = meta or {}
        hits.append({
            "text": safe_str(doc),
            "meta": meta,
            "distance": float(dist) if dist is not None else None,
            "tokens": int(meta.get("chunk_tokens", approximate_count_tokens(doc))),
        })
    return hits

def build_rag_context(question: str, top_k: int = TOP_K, max_context_tokens: int = MAX_CONTEXT_TOKENS):
    hits = retrieve_chunks(question, top_k=top_k)
    context_text, used = truncate_to_token_budget(hits, max_context_tokens)
    return context_text, used

# Load LLM baselines (Groq API)

In [ ]:
import os
from groq import Groq

LLM1_MODEL = "llama-3.3-70b-versatile"   # 70B params
LLM2_MODEL = "qwen/qwen3-32b"            # 32B params

# --- Google Colab (use when running in Colab) ---
# from google.colab import userdata
# groq_client = Groq(api_key=userdata.get("GROQ_API_KEY"))

# --- Local WSL2 (use when running locally) ---
# Set GROQ_API_KEY env var before launching Jupyter:
#   export GROQ_API_KEY="your_key_from_console.groq.com/keys"
groq_client = Groq(api_key=os.environ.get("GROQ_API_KEY"))

print(f"Groq client ready. Models: {LLM1_MODEL}, {LLM2_MODEL}")

# Unified generation

In [ ]:
STOP_MARKERS = [
    "<|im_end|>", "<|im_start|>",
    "<reponame>", "<gh_stars>",
]

def cut_garbage(text: str) -> str:
    text = (text or "").strip()
    for m in STOP_MARKERS:
        i = text.find(m)
        if i != -1:
            text = text[:i].strip()
            break
    return text

def gen_phi(model, question: str, context: str = "", max_new_tokens=192) -> str:
    if context.strip():
        user_content = (
            "Use the following course material excerpts to answer.\n\n"
            f"COURSE MATERIAL:\n{context}\n\n"
            f"QUESTION:\n{question}"
        )
    else:
        user_content = question

    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": user_content},
    ]

    device = "cuda" if torch.cuda.is_available() else "cpu"
    input_ids = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to(device)

    attn = torch.ones_like(input_ids)
    eos = tokenizer.eos_token_id
    pad = eos if tokenizer.pad_token_id is None else tokenizer.pad_token_id

    with torch.inference_mode():
        out = model.generate(
            input_ids=input_ids,
            attention_mask=attn,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.1,
            eos_token_id=eos,
            pad_token_id=pad,
            use_cache=True,
        )

    gen_ids = out[0][input_ids.shape[1]:]
    text = tokenizer.decode(gen_ids, skip_special_tokens=True)
    return cut_garbage(text)

def gen_llm(model_name: str, question: str, max_tokens: int = 192) -> str:
    """Generate answer via Groq API."""
    response = groq_client.chat.completions.create(
        model=model_name,
        messages=[
            {"role": "system", "content": SYSTEM},
            {"role": "user", "content": question},
        ],
        max_tokens=max_tokens,
        temperature=0,
    )
    return (response.choices[0].message.content or "").strip()

def pred_base(q: str) -> str:
    return gen_phi(base_model, q, context="")

def pred_lora(q: str) -> str:
    return gen_phi(lora_model, q, context="")

def pred_lora_rag(q: str) -> tuple[str, list]:
    ctx, used = build_rag_context(q)
    ans = gen_phi(lora_model, q, context=ctx)
    return ans, used

def pred_llm1(q: str) -> str:
    ans = gen_llm(LLM1_MODEL, q)
    time.sleep(2)  # respect 30 RPM rate limit
    return ans

def pred_llm2(q: str) -> str:
    ans = gen_llm(LLM2_MODEL, q)
    time.sleep(2)  # respect 30 RPM rate limit
    return ans

# Compute metrics (BLEU-2, ROUGE-L, METEOR, BERTScore)

In [ ]:
import nltk
from sacrebleu.metrics import BLEU

nltk.download("wordnet")
nltk.download("omw-1.4")

def bleu2_corpus(preds, refs) -> float:
    """Corpus-level BLEU-2 score (compatible with Colab sacrebleu)."""
    bleu2 = BLEU(max_ngram_order=2)
    score = bleu2.corpus_score(preds, [refs]).score
    return float(score)

# Run evaluation (SLM-base vs SLM-LoRA vs SLM-LoRA-RAG vs LLMs)

In [ ]:
def run_eval(df: pd.DataFrame, limit: int | None = None) -> pd.DataFrame:
    recs = []
    n = len(df) if limit is None else min(limit, len(df))

    for i in range(n):
        q = df.loc[i, "question"]
        ref = df.loc[i, "reference"]

        p_base = pred_base(q)

        p_lora = pred_lora(q)

        p_lora_rag, used_chunks = pred_lora_rag(q)

        p_llm1 = pred_llm1(q)

        p_llm2 = pred_llm2(q)

        recs.append({
            "idx": i,
            "question": q,
            "reference": ref,
            "pred_base": p_base,
            "pred_lora": p_lora,
            "pred_lora_rag": p_lora_rag,
            "pred_llm1": p_llm1,
            "pred_llm2": p_llm2,
            "rag_used_k": len(used_chunks),
            "rag_avg_dist": float(np.mean([h.get("distance") for h in used_chunks if h.get("distance") is not None])) if used_chunks else None,
        })

        if (i + 1) % 10 == 0:
            print(f"Done {i+1}/{n}")

    return pd.DataFrame(recs)

df_out = run_eval(df_eval, limit=100)
df_out.head(2)

# Summarize table

In [ ]:
import evaluate

rouge = evaluate.load("rouge")
meteor = evaluate.load("meteor")
bertscore = evaluate.load("bertscore")


In [ ]:
def summarize(df: pd.DataFrame) -> pd.DataFrame:
    rows = []

    configs = [
        ("BASE", "pred_base"),
        ("LoRA", "pred_lora"),
        ("LoRA+RAG", "pred_lora_rag"),
        ("LLM (Llama-3.3-70B)", "pred_llm1"),
        ("LLM (Qwen3-32B)", "pred_llm2"),
    ]

    for model_name, pred_col, lat_col in configs:
        preds = df[pred_col].tolist()
        refs  = df["reference"].tolist()

        rows.append({
            "model": model_name,
            "BLEU": bleu2_corpus(preds, refs),
            "ROUGE-L": rouge.compute(predictions=preds, references=refs)["rougeL"],
            "METEOR": meteor.compute(predictions=preds, references=refs)["meteor"],
            "BERTScore_F1": float(np.mean(
                bertscore.compute(
                    predictions=preds, references=refs,
                    lang="en", rescale_with_baseline=True,
                )["f1"]
            )),
            "latency_mean_s": float(df[lat_col].mean()),
            "latency_median_s": float(df[lat_col].median()),
        })

    return pd.DataFrame(rows)

df_summary = summarize(df_out)
df_summary

# Save CSV artifacts

In [ ]:
OUT_DIR = f"{BASE_DIR}/metrics_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

pred_path = f"{OUT_DIR}/{SUBJECT}_preds_all_models.csv"
sum_path  = f"{OUT_DIR}/{SUBJECT}_metrics_all_models.csv"

df_out.to_csv(pred_path, index=False)
df_summary.to_csv(sum_path, index=False)

print("Saved:")
print(pred_path)
print(sum_path)